# 05 — Train MULTIVARIATE AE & upload to KQL `models`

Multivariate sibling of `04_train_univariate_ae.ipynb`:

1. Read 8 sensors of **one machine** from the wide MV `raw_telemetry_wide_mv` (forward-filled).
2. Build **sliding** windows of `WINDOW_SIZE` bins over an `(T, 8)` matrix.
3. Train a small LSTM autoencoder in PyTorch.
4. Export a `ScoreWrapper` to **ONNX** (output = MSE per window across all sensors).
5. Compute `threshold = mean(loss) + 3 * std(loss)` on the training set and store it in `metadata.threshold`.
6. Upload to the `models` table as a new `version` of `multivariate_ae__M-001`.
7. Smoke-test by calling `score_multivariate_onnx_batch_from_mv(...)` directly in KQL.

Prereqs: `kql/05_multivariate_mv.kql` already applied (MV exists and is being maintained), simulator running, `python` plugin enabled.

Runs both **locally** (`.venv`, device-code auth) and **inside Fabric** (auto-detected via `notebookutils`).

## 1. Config

In [1]:
MACHINE      = "M-001"
# IMPORTANT: order MUST match `pack_array(...)` in build_multivariate_windows_batch_from_mv
# (file: kql/05_multivariate_mv.kql).
SENSORS      = [
    "temperature_motor",
    "temperature_bearing",
    "vibration_axial",
    "vibration_radial",
    "current",
    "power",
    "spindle_rpm",
    "pressure_hydraulic",
]
WINDOW_SIZE  = 64           # 64 bins of 1s = 64s per window
BIN_SIZE     = "1s"
LOOKBACK     = "30m"        # KQL timespan literal for training data
EPOCHS       = 50
HIDDEN       = 64
MODEL_NAME   = f"multivariate_ae__{MACHINE}"
THRESHOLD_K  = 3.0          # threshold = mean(loss) + K * std(loss) on training set

## 2. Auth + Kusto client (dual mode: local `.venv` or Fabric notebook)

In [2]:
import base64
import datetime as dt
import io
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from azure.kusto.data import KustoClient, KustoConnectionStringBuilder
from azure.kusto.data.helpers import dataframe_from_result_table

FABRIC_API   = "https://api.fabric.microsoft.com/v1"
FABRIC_SCOPE = "https://api.fabric.microsoft.com/.default"

try:
    import notebookutils  # only present inside Fabric runtime
    IN_FABRIC = True
except ImportError:
    IN_FABRIC = False

if IN_FABRIC:
    from azure.identity import DefaultAzureCredential
    ctx        = notebookutils.runtime.context
    WORKSPACE  = ctx.get("currentWorkspaceName")
    TENANT_ID  = ctx.get("tenantId", "")
    KQLDB_NAME = os.environ.get("FABRIC_KQLDB_NAME", "kql_telemetry")
    cred       = DefaultAzureCredential()
    print(f"[auth] Fabric runtime (workspace={WORKSPACE})")
else:
    from dotenv import load_dotenv
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / ".env").exists() and REPO_ROOT.parent != REPO_ROOT:
        REPO_ROOT = REPO_ROOT.parent
    load_dotenv(REPO_ROOT / ".env")
    sys.path.insert(0, str(REPO_ROOT / "tools"))
    from _fabric_auth import get_credential  # noqa: E402
    TENANT_ID  = os.environ["FABRIC_TENANT_ID"]
    WORKSPACE  = os.environ["FABRIC_WORKSPACE_NAME"]
    KQLDB_NAME = os.environ["FABRIC_KQLDB_NAME"]
    cred       = get_credential(TENANT_ID, FABRIC_SCOPE, REPO_ROOT)
    print(f"[auth] local .venv (.env @ {REPO_ROOT})")

fabric_token = cred.get_token(FABRIC_SCOPE).token
headers      = {"Authorization": f"Bearer {fabric_token}"}

ws = next(w for w in requests.get(f"{FABRIC_API}/workspaces", headers=headers).json()["value"]
          if w["displayName"] == WORKSPACE)
ws_id = ws["id"]
db    = next(d for d in requests.get(f"{FABRIC_API}/workspaces/{ws_id}/kqlDatabases", headers=headers).json()["value"]
             if d["displayName"] == KQLDB_NAME)
query_uri = db["properties"]["queryServiceUri"]
print("queryServiceUri:", query_uri)

kcsb  = KustoConnectionStringBuilder.with_azure_token_credential(query_uri, cred)
kusto = KustoClient(kcsb)

[auth] using cached credentials
[auth] local .venv (.env @ C:\Users\faustopalma\OneDrive - Microsoft\Documents\Customers\Other\Iveco\anomalydetection)


queryServiceUri: https://trd-xr97y56tuzzkxy5cgp.z5.kusto.fabric.microsoft.com


## 3. Pull wide data and build sliding windows

In [3]:
cols_kql = ", ".join(SENSORS)
q = f"""
raw_telemetry_wide_mv
| where machine_id == '{MACHINE}'
| where ts_bin > now() - {LOOKBACK}
| order by ts_bin asc
| project ts_bin, {cols_kql}
"""
res = kusto.execute(KQLDB_NAME, q)
df  = dataframe_from_result_table(res.primary_results[0])
print("raw rows from MV:", len(df))
df.head()

raw rows from MV: 242


,ts_bin,temperature_motor,temperature_bearing,vibration_axial,vibration_radial,current,power,spindle_rpm,pressure_hydraulic
0,2026-05-14 17:39:42+00:00,61.4944,56.7327,0.2555,0.3594,12.58,8.2354,3112.187,119.7278
1,2026-05-14 17:39:43+00:00,60.4208,56.8405,0.248,0.3145,12.601,8.3134,3114.0314,121.4134
2,2026-05-14 17:39:45+00:00,61.6229,56.7521,0.1582,0.2431,12.509,7.9617,3103.8834,121.1345
3,2026-05-14 17:39:46+00:00,60.5328,56.3766,0.1844,0.2127,11.6211,7.9207,3113.7559,121.0552
4,2026-05-14 17:39:47+00:00,61.2512,56.5229,0.1422,0.2362,11.9343,7.9121,3106.9805,122.3165


In [4]:
# Forward-fill any NULL cells per sensor (industrial-historian standard).
# Keeps SENSORS ordering, drops still-NaN initial rows (no value yet).
df_ff = df.set_index("ts_bin")[SENSORS].ffill().dropna(how="any")
values = df_ff.to_numpy(dtype=np.float32)
print("after ffill+dropna:", values.shape)

# Sliding windows of WINDOW_SIZE bins, stride 1.
T, F = values.shape
if T < WINDOW_SIZE + 4:
    raise RuntimeError(f"Not enough bins ({T}) to train. Increase LOOKBACK or wait for more data.")
n_win = T - WINDOW_SIZE + 1
X = np.lib.stride_tricks.sliding_window_view(values, (WINDOW_SIZE, F))[:, 0, :, :]
X = np.ascontiguousarray(X.astype(np.float32))
print("training tensor:", X.shape, "(N_windows, window_size, n_features)")

after ffill+dropna: (242, 8)
training tensor: (179, 64, 8) (N_windows, window_size, n_features)


## 4. Model + training

In [5]:
import torch
from torch import nn

torch.manual_seed(0)

class WindowAE(nn.Module):
    def __init__(self, n_features: int, hidden: int = HIDDEN):
        super().__init__()
        self.enc = nn.LSTM(n_features, hidden, batch_first=True)
        self.dec = nn.LSTM(hidden, n_features, batch_first=True)
    def forward(self, x):
        h, _   = self.enc(x)
        out, _ = self.dec(h)
        return out

model = WindowAE(n_features=F)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
loss  = nn.MSELoss()

t = torch.from_numpy(X)
for epoch in range(EPOCHS):
    opt.zero_grad()
    e = loss(model(t), t)
    e.backward()
    opt.step()
    if epoch % 5 == 0 or epoch == EPOCHS - 1:
        print(f"epoch {epoch:>3}  loss {e.item():.6f}")

final_loss = float(e.item())

epoch   0  loss 1196822.000000


epoch   5  loss 1196733.875000


epoch  10  loss 1196694.625000


epoch  15  loss 1196670.375000


epoch  20  loss 1196649.500000


...

epoch  30  loss 1196612.125000


epoch  35  loss 1196592.625000


epoch  40  loss 1196570.125000


epoch  45  loss 1196542.000000


epoch  49  loss 1196515.125000


## 5. Export to ONNX (with `ScoreWrapper`) + compute threshold

The wrapper returns a single scalar score per window: MSE averaged over
**both** the time axis and the feature axis. The threshold is computed
from the per-window losses on the training set so the scoring function
in KQL can read it from `metadata.threshold` without a hard-coded value.

In [6]:
class ScoreWrapper(nn.Module):
    def __init__(self, ae: nn.Module):
        super().__init__()
        self.ae = ae
    def forward(self, x):
        recon = self.ae(x)
        return ((recon - x) ** 2).mean(dim=(1, 2))

wrapped = ScoreWrapper(model).eval()
dummy   = torch.zeros((1, WINDOW_SIZE, F), dtype=torch.float32)

buf = io.BytesIO()
torch.onnx.export(
    wrapped, dummy, buf,
    input_names=["window"], output_names=["score"],
    dynamic_axes={"window": {0: "batch"}, "score": {0: "batch"}},
    opset_version=17,
)
onnx_bytes = buf.getvalue()

# Sandbox onnxruntime supports IR <= 9.
import onnx
mp = onnx.load_from_string(onnx_bytes)
if mp.ir_version > 9:
    mp.ir_version = 9
    onnx_bytes = mp.SerializeToString()

onnx_b64 = base64.b64encode(onnx_bytes).decode("ascii")
print(f"ONNX size: {len(onnx_bytes)/1024:.1f} KB  (base64 {len(onnx_b64)/1024:.1f} KB), ir={mp.ir_version}")

C:\Users\faustopalma\AppData\Local\Temp\ipykernel_43324\2222123167.py:13: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0514 19:44:31.657000 43324 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


W0514 19:44:32.720000 43324 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:110] torchvision is not installed. Skipping torchvision::nms


W0514 19:44:32.723000 43324 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:110] torchvision is not installed. Skipping torchvision::roi_align


W0514 19:44:32.725000 43324 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:110] torchvision is not installed. Skipping torchvision::roi_pool


[torch.onnx] Obtain model graph for `ScoreWrapper([...]` with `torch.export.export(..., strict=False)`...


...

C:\Users\faustopalma\AppData\Local\Programs\Python\Python313\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "C:\Users\faustopalma\OneDrive - Microsoft\Documents\Customers\Other\Iveco\anomalydetection\.venv\Lib\site-packages\onnxscript\version_converter\__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "C:\Users\faustopalma\OneDrive - Microsoft\Documents\Customers\Other\Iveco\anomalydetection\.venv\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "C:\Users\faustopalma\OneDrive - Microsoft\Documents\Customers\Other\Iveco\anomalydetection\.venv\Lib\site-packages\onnxscript\version_converter\__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_ver

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...


[torch.onnx] Optimize the ONNX graph... ✅
ONNX size: 129.9 KB  (base64 173.2 KB), ir=9


In [7]:
# Per-window scores on the training set -> threshold = mean + K*std.
import onnxruntime as ort
sess = ort.InferenceSession(onnx_bytes)
scores_train = sess.run(None, {"window": X})[0]
threshold = float(scores_train.mean() + THRESHOLD_K * scores_train.std())
print(f"train scores: mean={scores_train.mean():.6f}  std={scores_train.std():.6f}  -> threshold={threshold:.6f} (K={THRESHOLD_K})")
print("sample scores:", scores_train[:3])

train scores: mean=1196507.500000  std=2513.622314  -> threshold=1204048.375000 (K=3.0)
sample scores: [1.1973215e+06 1.1968248e+06 1.1962905e+06]


## 6. Upload into the `models` table

In [8]:
next_v = 1
try:
    res = kusto.execute(KQLDB_NAME, f"models | where name == '{MODEL_NAME}' | summarize v = max(version)")
    cur = dataframe_from_result_table(res.primary_results[0])["v"].iloc[0]
    if cur is not None and not pd.isna(cur):
        next_v = int(cur) + 1
except Exception as ex:
    print("warn: lookup version failed, defaulting to 1:", ex)

created_at = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
metadata   = json.dumps({
    "final_loss":     final_loss,
    "n_windows":      int(X.shape[0]),
    "trained_at":     created_at,
    "machine":        MACHINE,
    "sensors":        SENSORS,
    "bin_size":       BIN_SIZE,
    "threshold":      threshold,
    "threshold_k":    THRESHOLD_K,
    "hidden":         HIDDEN,
})

sensors_dyn = json.dumps(SENSORS)
cmd = (
    ".set-or-append models <|\n"
    "print "
    f"name='{MODEL_NAME}', "
    f"version=int({next_v}), "
    f"created_at=datetime({created_at}), "
    "framework='onnx', "
    f"window_size=int({WINDOW_SIZE}), "
    f"sensors=todynamic('{sensors_dyn}'), "
    f"payload='{onnx_b64}', "
    f"metadata=todynamic('{metadata}')"
)
kusto.execute_mgmt(KQLDB_NAME, cmd)
print(f"uploaded {MODEL_NAME} v{next_v} (threshold={threshold:.6f})")

uploaded multivariate_ae__M-001 v1 (threshold=1204048.375000)


## 7. Smoke test in KQL

In [9]:
smoke = f"""
score_multivariate_onnx_batch_from_mv('{MODEL_NAME}', '{MACHINE}')
| top 5 by window_end desc
| project window_start, window_end, score, is_anomaly, model_version
"""
res = kusto.execute(KQLDB_NAME, smoke)
dataframe_from_result_table(res.primary_results[0])

,window_start,window_end,score,is_anomaly,model_version
0,2026-05-14 17:41:51+00:00,2026-05-14 17:42:54+00:00,1199941.625,False,1
1,2026-05-14 17:40:47+00:00,2026-05-14 17:41:50+00:00,1193803.5,False,1
2,2026-05-14 17:39:42+00:00,2026-05-14 17:40:46+00:00,1197321.5,False,1
